In [1]:
import pandas as pd
import hashlib
import requests
import matplotlib.pyplot as plt



In [2]:
import pandas as pd
import requests
import hashlib

response = requests.get(
    "https://datasets-server.huggingface.co/parquet",
    params={"dataset": "electricsheepafrica/Nigerian-Financial-Transactions-and-Fraud-Detection-Dataset"}
)

parquet_files = response.json()["parquet_files"]

TARGET_ROWS = 20000
sampled_rows = []
total_kept = 0

for file_info in parquet_files:
    if total_kept >= TARGET_ROWS:
        break
    
    print(f"Loading {file_info['url'].split('/')[-1]}...")
    transactions_data = pd.read_parquet(file_info["url"])
    
    # Filter for Lagos
    transactions_data_lagos = transactions_data[transactions_data['location'] == 'Lagos']
    print(f"  Lagos rows in file: {len(transactions_data):,}")
    
    for idx, row in transactions_data_lagos.iterrows():
        if total_kept >= TARGET_ROWS:
            break
            
        unique_string = f"{row['transaction_id']}_{row['is_fraud']}"
        hash_value = int(hashlib.md5(unique_string.encode()).hexdigest(), 16) % 100
        
        if hash_value < 4:  # 4% from each file
            sampled_rows.append(row.to_dict())
            total_kept += 1
    
    print(f"  Total kept so far: {total_kept:,}\n")

final_transactions_data = pd.DataFrame(sampled_rows)
print(f"Collected {len(final_transactions_data):,} Lagos rows")
final_transactions_data.to_csv("lagos_20k.csv", index=False)
print("Saved to lagos_20k.csv")

Loading 0000.parquet...
  Lagos rows in file: 1,220,000
  Total kept so far: 4,762

Loading 0001.parquet...
  Lagos rows in file: 1,220,000
  Total kept so far: 9,589

Loading 0002.parquet...
  Lagos rows in file: 1,220,000
  Total kept so far: 14,350

Loading 0003.parquet...
  Lagos rows in file: 1,220,000
  Total kept so far: 19,146

Loading 0004.parquet...
  Lagos rows in file: 120,000
  Total kept so far: 19,590

Collected 19,590 Lagos rows
Saved to lagos_20k.csv
